# SPECTRA-Siam — CodeXGLUE (BigCloneBench) V3 — Topology + canonical labels + lexical sketches (main method)

The full proposed input: exported topology, canonical node labels, and hashed lexical node sketches enter only the encoder.

The classifier consumes only separate symmetric comparisons of density, heat-trace, and Chebyshev spectral blocks.


In [ ]:
# Kaggle configuration.  Change CLEAN_DATA_DIR only if automatic discovery
# does not find the attached XGLUE clean-data folder.
from __future__ import annotations

import json
import math
import random
import time
import zipfile
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch


def require_supported_kaggle_gpu() -> None:
    """Fail early on accelerators unsupported by Kaggle's current PyTorch build."""
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU is enabled. In Kaggle select GPU accelerator: 2x T4.")
    capability = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    if capability[0] < 7:
        raise RuntimeError(
            f"{name} has CUDA capability sm_{capability[0]}{capability[1]}, which the current Kaggle PyTorch "
            "build cannot execute. Select 2x T4 in Kaggle Accelerator settings, restart the session, and Run All."
        )
    print({"gpu": name, "capability": f"sm_{capability[0]}{capability[1]}", "gpu_count": torch.cuda.device_count()})


require_supported_kaggle_gpu()

SEED = 42
DATASET_KEY = "codexglue"
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# Preferred Kaggle attachment layout.  The next cell safely discovers another
# attached `clean_data` directory if this exact path does not exist.
CLEAN_DATA_DIR = KAGGLE_INPUT / DATASET_KEY

# Execution flags requested for portable Kaggle use.
# Runtime profile.  Use quick_1h for preliminary results; change only this
# value to extended_6_7h for the larger follow-up run.
RUN_PROFILE = "final_full"
RUN_PRESETS = {
    "diagnostic_10k": {"max_train_pairs": 10_000, "max_valid_pairs": 2_000, "max_test_pairs": 2_000, "epochs": 5},
    "quick_1h": {"max_train_pairs": 75_000, "max_valid_pairs": 10_000, "max_test_pairs": 10_000, "epochs": 4},
    "extended_6_7h": {"max_train_pairs": 650_000, "max_valid_pairs": 80_000, "max_test_pairs": 80_000, "epochs": 8},
}
# Use this profile in every method notebook for a data-equal comparison.
RUN_PRESETS["comparison_50k"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": 50_000,
    "max_valid_pairs": 10_000,
    "max_test_pairs": 10_000,
    # 50k x 6 gives the same number of pair presentations as quick_1h: 75k x 4.
    "epochs": 6,
}

# Final paper protocol: every graph-evaluable pair in each official split.
# Four epochs are retained from the validated full-data SPECTRA schedule.
RUN_PRESETS["final_full"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": None,
    "max_valid_pairs": None,
    "max_test_pairs": None,
    "epochs": 4,
}

# --- bounded run budget ---
# SPECTRA-Siam is the slowest model in the table (3.5h on BigCloneBench at full
# data) and its cost grows with graph size, which the merge fix increased on
# AtCoder. Fewer, larger epochs keep the number of pair presentations close to
# the published run while bounding wall time.
RUN_PRESETS["bounded_10h"] = {
    **RUN_PRESETS["comparison_50k"],
    "max_train_pairs": 200_000,
    "max_valid_pairs": 20_000,
    "max_test_pairs": 20_000,
    "epochs": 4,
}

if RUN_PROFILE not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose one of {tuple(RUN_PRESETS)}")
RUN_CONFIG = RUN_PRESETS[RUN_PROFILE]
RUN_DIAGNOSTICS = RUN_PROFILE == "diagnostic_10k"
MAX_TRAIN_PAIRS = RUN_CONFIG["max_train_pairs"]
MAX_VALID_PAIRS = RUN_CONFIG["max_valid_pairs"]
MAX_TEST_PAIRS = RUN_CONFIG["max_test_pairs"]
RUN_EXPERIMENT = True
USE_AMP = True

# Multi-relational structural input: AST syntax plus DDG projected onto AST nodes.
# Relation 1 is a sequential fallback and is deliberately disabled here.
INPUT_RELATION_INDICES = (0, 2)  # AST + DDG
MAX_AST_NODES = 256
HIDDEN_DIM = 256
LATENT_NODES = 32
SLOT_ITERATIONS = 3
ATTENTION_HEADS = 4
LEXICAL_DROPOUT = 0.30
USE_NODE_LEXICAL = True
# A language-neutral source-token residual is enabled only for AtCoder's
# Java--Python pairs. It is not a raw AST-node-label vocabulary.
# Raw source tokens stay out of every reported run: the structural claim has to
# be measured on structure. 04_feature_ablation overrides this per arm, which is
# the one place the residual is the subject of the experiment rather than a
# silent advantage on a single benchmark.
USE_SOURCE_LEXICAL = False
# Node states may induce the latent graph, but only eigenvalue-derived
# density and heat features reach the paper-facing code readout.
READOUT_MODE = "graph_signal_spectral"
# Topology-only ablation: collapse every canonical category to a single id so
# the encoder sees adjacency and nothing else. False for all normal runs.
STRIP_NODE_TYPES = False
# Input ablation only: remove every exported edge before encoder propagation.
STRIP_TOPOLOGY = False
SOURCE_LEXICAL_SLOTS = 96
TARGET_DENSITY = 0.15
CHEBYSHEV_DEGREE = 12
EPOCHS = RUN_CONFIG["epochs"]
# --- session time guard ---
# Hard wall for one Kaggle session, in hours. Training stops after the current
# epoch when the next one is projected to cross it, and the best checkpoint so
# far is evaluated. Set to None to disable.
SESSION_BUDGET_HOURS = 8.5
BATCH_SIZE = 32
EFFECTIVE_BATCH_SIZE = 128
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
POSITIVE_CLASS_WEIGHT = "auto"  # "auto" computes neg/pos from Java train
SPECTRAL_WEIGHT = 0.30
SPECTRAL_VARIANCE_FLOOR = 0.03
RECONSTRUCTION_WEIGHT = 0.05
GRAPH_WEIGHT = 0.01
# AtCoder pairs are exclusively Java--Python.  This aligns code embeddings
# directly while preserving the graph/spectral losses below.
# Was applied to AtCoder only, which made its loss function different from every
# other column in the table.
EMBEDDING_CONTRASTIVE_WEIGHT = 0.0
AUC_RANKING_WEIGHT = 0.10
EMBEDDING_NEGATIVE_MARGIN = 0.20
# Batch-hard mining over known non-clone pairs.
HARD_NEGATIVE_WEIGHT = 0.20
HARD_NEGATIVE_MARGIN = 0.10
HARD_NEGATIVE_FRACTION = 0.25
# Complexity-conditioned, learnable latent-adjacency temperature.
ADAPTIVE_TEMPERATURE_MIN = 0.20
ADAPTIVE_TEMPERATURE_MAX = 1.20
USE_ADAPTIVE_TEMPERATURE = True
USE_SPECTRAL_ATTENTION = True
NUM_WORKERS = 0
TEMPERATURE_START, TEMPERATURE_END = 1.0, 0.30  # retained only for backward-compatible checkpoints

REQUIRED_CLEAN_FILES = {
    "codes": ("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp"),
    "pairs": ("pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp"),
    # Kaggle sometimes unpacks a file named graph_spectra.jsonl.gz.tmp into
    # graph_spectra.jsonl/graph_spectra.jsonl.gz.tmp.  Support that layout.
    "graphs": ("graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp"),
}

def has_artifact(path: Path, names: tuple[str, ...]) -> bool:
    return any((path / name).is_file() for name in names) or any(any(path.rglob(name)) for name in names)

def is_complete_clean_data(path: Path) -> bool:
    return path.is_dir() and all(has_artifact(path, names) for names in REQUIRED_CLEAN_FILES.values())

def find_complete_clean_data(search_root: Path) -> list[Path]:
    roots = {search_root} if search_root.exists() else set()
    graph_names = REQUIRED_CLEAN_FILES["graphs"]
    for name in graph_names:
        for graph_path in search_root.rglob(name):
            # Test the graph folder, its clean_data parent, and higher parents,
            # but never scan outside /kaggle/input.
            for parent in graph_path.parents:
                if parent == search_root or search_root in parent.parents:
                    roots.add(parent)
    return [path for path in roots if is_complete_clean_data(path)]

def extract_clean_data_zip(search_root: Path) -> list[Path]:
    """Extract only when an attached ZIP actually contains all clean artifacts."""
    extracted = []
    for archive_path in search_root.rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                names = set(archive.namelist())
                if not any(name.endswith("graph_spectra.jsonl.gz") or name.endswith("graph_spectra.jsonl") for name in names):
                    continue
                if not any(name.endswith("pairs.csv.gz") or name.endswith("pairs.csv") for name in names):
                    continue
                if not any(name.endswith("codes.jsonl.gz") or name.endswith("codes.jsonl") for name in names):
                    continue
                target = WORK_DIR / "attached_clean_data" / archive_path.stem
                marker = target / ".extracted"
                if not marker.exists():
                    target.mkdir(parents=True, exist_ok=True)
                    archive.extractall(target)
                    marker.write_text("ok", encoding="utf-8")
                extracted.extend(find_complete_clean_data(target))
        except zipfile.BadZipFile:
            continue
    return extracted

def resolve_clean_data(path: Path) -> Path:
    candidates = [path] if is_complete_clean_data(path) else []
    candidates.extend(find_complete_clean_data(KAGGLE_INPUT))
    if not candidates:
        candidates.extend(extract_clean_data_zip(KAGGLE_INPUT))
    candidates = sorted(set(candidates), key=lambda item: (DATASET_KEY.lower() not in str(item).lower(), -len(item.parts), str(item)))
    if candidates:
        return candidates[0]
    attached = [str(item.relative_to(KAGGLE_INPUT)) for item in KAGGLE_INPUT.rglob("*") if item.is_file()][:80]
    raise FileNotFoundError(
        "A complete clean-data attachment is required: codes.jsonl(.gz), pairs.csv(.gz), and "
        "graph_spectra.jsonl(.gz). The current attachment has no graph_spectra file. Re-upload the final "
        "clean_data ZIP/folder produced by the graph-export pipeline. Files seen under /kaggle/input: "
        + ", ".join(attached)
    )

CLEAN_DATA_DIR = resolve_clean_data(CLEAN_DATA_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")
print({"dataset": str(CLEAN_DATA_DIR), "device": str(DEVICE), "amp": AMP_ENABLED, "torch": torch.__version__})

METHOD_VARIANT = "input_only_lex"








INPUT_ABLATION = "lex"
RUN_TAG = f"{DATASET_KEY}_{METHOD_VARIANT}"
# Inputs affect only latent-graph induction. The pair head receives no raw input, embedding, or lexical bypass.


## Canonical type mapping

Raw parser/Joern names are reduced to a fixed cross-language vocabulary before embedding.

In [ ]:
"""Language-agnostic canonical node types for code graphs.

Parser node names are an implementation detail: the same construct may be an
``If``, ``if_statement`` or ``CONTROL_STRUCTURE_IF`` depending on whether the
graph came from Python's :mod:`ast`, tree-sitter, or Joern.  This module maps
those heterogeneous names to a small, stable vocabulary suitable for a shared
embedding table.

The mapper deliberately consumes *node types*, not source-code tokens.  It is
also conservative: ambiguous tags such as bare ``LITERAL`` and
``CONTROL_STRUCTURE`` map to their explicit generic categories, rather than
being guessed as a numeric literal or a particular branch construct.

IDs are stable, contiguous, and zero-based.  If a downstream model reserves
zero for padding, store ``get_canonical_id(raw_type) + 1`` and allocate
``NUM_CANONICAL_TYPES + 1`` embeddings.
"""

from __future__ import annotations

import re
import html
from collections.abc import Iterable
from typing import Final


# Do not reorder this tuple: its position is the serialized embedding ID.
CANONICAL_CATEGORIES: Final[tuple[str, ...]] = (
    "Control_If",
    "Control_Loop",
    "Control_Switch",
    "Control_Return",
    "Control_Break",
    "Var_Decl",
    "Func_Decl",
    "Class_Decl",
    "Param_Decl",
    "Assign_Op",
    "Binary_Op",
    "Unary_Op",
    "Call_Expr",
    "Access_Expr",
    "Literal_Num",
    "Literal_Str",
    "Literal_Bool",
    "Literal_Null",
    "Structural_Block",
    "Identifier_Context",
    "Literal_Generic",
    "Control_Generic",
    "Type_Meta",
    "Canonical_Unknown",
)

CANONICAL_TYPE_TO_ID: Final[dict[str, int]] = {
    name: index for index, name in enumerate(CANONICAL_CATEGORIES)
}
CANONICAL_ID_TO_TYPE: Final[tuple[str, ...]] = CANONICAL_CATEGORIES
NUM_CANONICAL_TYPES: Final[int] = len(CANONICAL_CATEGORIES)
CANONICAL_UNKNOWN_ID: Final[int] = CANONICAL_TYPE_TO_ID["Canonical_Unknown"]


_CAMEL_WORD_BOUNDARY = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")
_ACRONYM_WORD_BOUNDARY = re.compile(r"(?<=[A-Z])(?=[A-Z][a-z])")
_NON_ALNUM = re.compile(r"[^a-z0-9]+")


def _normalize(raw_type: object) -> str:
    """Normalize parser spelling without reading source-level token content."""
    if not isinstance(raw_type, str):
        return ""
    text = html.unescape(raw_type).strip()
    if not text:
        return ""
    text = _ACRONYM_WORD_BOUNDARY.sub("_", text)
    text = _CAMEL_WORD_BOUNDARY.sub("_", text)
    return _NON_ALNUM.sub("_", text.lower()).strip("_")


# Exact aliases cover the common output of Joern, Python ast, JavaParser,
# srcML, clang/tree-sitter C, and tree-sitter Python/Java.  Keys are normalized
# once at import time, so lookup remains inexpensive inside a DataLoader.
_ALIASES_BY_CATEGORY: Final[dict[str, tuple[str, ...]]] = {
    "Control_If": (
        "operator_conditional",
        "if",
        "if_stmt",
        "if_statement",
        "if_expression",
        "if_exp",
        "if_then_statement",
        "if_then_else_statement",
        "conditional_expression",
        "ternary_expression",
        "control_structure_if",
    ),
    "Control_Loop": (
        "for",
        "for_stmt",
        "for_statement",
        "for_each",
        "for_each_statement",
        "foreach_statement",
        "enhanced_for_statement",
        "async_for",
        "while",
        "while_stmt",
        "while_statement",
        "do",
        "do_stmt",
        "do_statement",
        "do_while_statement",
        "loop_statement",
        "list_comp",
        "set_comp",
        "comprehension",
        "dict_comp",
        "generator_exp",
        "control_structure_for",
        "control_structure_while",
        "control_structure_do",
    ),
    "Control_Switch": (
        "switch",
        "switch_stmt",
        "switch_statement",
        "switch_expression",
        "switch_entry",
        "case",
        "case_statement",
        "case_label",
        "match",
        "match_statement",
        "match_case",
        "control_structure_switch",
    ),
    "Control_Return": (
        "return",
        "return_stmt",
        "return_statement",
        "yield",
        "yield_expr",
        "yield_expression",
        "control_structure_return",
    ),
    "Control_Break": (
        "break",
        "break_stmt",
        "break_statement",
        "continue",
        "continue_stmt",
        "continue_statement",
        "goto",
        "goto_statement",
        "control_structure_break",
        "control_structure_continue",
    ),
    "Var_Decl": (
        "local",
        "member",
        "field_declaration",
        "variable_declaration",
        "variable_declarator",
        "local_variable_declaration",
        "local_declaration_statement",
        "declaration",
        "declaration_statement",
        "init_declarator",
        "var_decl",
        "let_declaration",
        "const_declaration",
        "global",
        "nonlocal",
    ),
    "Func_Decl": (
        "method",
        "method_declaration",
        "method_definition",
        "function_declaration",
        "function_definition",
        "function_def",
        "async_function_def",
        "constructor_declaration",
        "constructor_definition",
        "lambda",
        "lambda_expression",
        "func_decl",
    ),
    "Class_Decl": (
        "type_decl",
        "class_decl",
        "class_declaration",
        "class_definition",
        "class_def",
        "interface_declaration",
        "annotation_type_declaration",
        "enum_declaration",
        "struct_specifier",
        "struct_declaration",
        "union_specifier",
        "record_declaration",
    ),
    "Param_Decl": (
        "keyword",
        "arg",
        "arguments",
        "parameter",
        "parameter_declaration",
        "formal_parameter",
        "receiver_parameter",
        "spread_parameter",
        "typed_parameter",
        "default_parameter",
        "method_parameter_in",
        "method_parameter_out",
        "param_decl",
    ),
    "Assign_Op": (
        "assign",
        "ann_assign",
        "aug_assign",
        "named_expr",
        "assignment",
        "assignment_expression",
        "assignment_operator",
        "operator_assignment",
        "operator_assignment_plus",
        "operator_assignment_minus",
        "operator_assignment_multiplication",
        "operator_assignment_division",
        "operator_assignment_modulo",
        "operator_assignment_and",
        "operator_assignment_or",
        "operator_assignment_xor",
        "operator_assignment_shift_left",
        "operator_assignment_arithmetic_shift_right",
        "operator_assignment_logical_shift_right",
    ),
    "Binary_Op": (
        "bin_op",
        "bool_op",
        "compare",
        "binary_expression",
        "binary_operator",
        "infix_expression",
        "add",
        "sub",
        "mult",
        "mat_mult",
        "div",
        "floor_div",
        "mod",
        "pow",
        "l_shift",
        "r_shift",
        "bit_or",
        "bit_xor",
        "bit_and",
        "and",
        "or",
        "eq",
        "not_eq",
        "lt",
        "lt_e",
        "gt",
        "gt_e",
        "is",
        "is_not",
        "in",
        "not_in",
        "operator_addition",
        "operator_subtraction",
        "operator_multiplication",
        "operator_division",
        "operator_modulo",
        "operator_exponentiation",
        "operator_shift_left",
        "operator_arithmetic_shift_right",
        "operator_logical_shift_right",
        "operator_and",
        "operator_or",
        "operator_xor",
        "operator_logical_and",
        "operator_logical_or",
        "operator_equals",
        "operator_not_equals",
        "operator_less_than",
        "operator_less_equals_than",
        "operator_greater_than",
        "operator_greater_equals_than",
    ),
    "Unary_Op": (
        "operator_size_of",
        "operator_cast",
        "unary_op",
        "unary_expression",
        "prefix_expression",
        "postfix_expression",
        "u_add",
        "u_sub",
        "not",
        "invert",
        "operator_not",
        "operator_logical_not",
        "operator_minus",
        "operator_plus",
        "operator_pre_increment",
        "operator_post_increment",
        "operator_pre_decrement",
        "operator_post_decrement",
        "operator_address_of",
        "operator_indirection",
    ),
    "Call_Expr": (
        "init",
        "operator_alloc",
        "operator_new",
        "operator_constructor",
        "call",
        "call_expr",
        "call_expression",
        "method_call",
        "method_call_expression",
        "method_invocation",
        "function_call",
        "function_call_expression",
        "object_creation_expression",
        "class_instance_creation_expression",
        "new_expression",
    ),
    "Access_Expr": (
        "operator_index_access",
        "slice",
        "identifier",
        "field_identifier",
        "name",
        "qualified_name",
        "attribute",
        "attribute_access",
        "member_access",
        "member_expression",
        "field_access",
        "field_expression",
        "array_access",
        "array_access_expression",
        "subscript",
        "index_expression",
        "pointer_expression",
        "this",
        "super",
        "external",
    ),
    "Literal_Num": (
        "num",
        "number",
        "number_literal",
        "numeric_literal",
        "integer_literal",
        "decimal_integer_literal",
        "hex_integer_literal",
        "octal_integer_literal",
        "binary_integer_literal",
        "floating_point_literal",
        "float_literal",
        "double_literal",
        "imaginary_literal",
        "literal_int",
        "literal_integer",
        "literal_float",
        "literal_double",
        "constant_int",
        "constant_integer",
        "constant_float",
        "constant_double",
        "constant_complex",
    ),
    "Literal_Str": (
        "str",
        "joined_str",
        "formatted_value",
        "string",
        "string_literal",
        "character_literal",
        "char_literal",
        "concatenated_string",
        "template_string",
        "literal_str",
        "literal_string",
        "literal_char",
        "constant_str",
        "constant_string",
        "constant_bytes",
    ),
    "Literal_Bool": (
        "bool",
        "boolean",
        "boolean_literal",
        "true",
        "false",
        "true_literal",
        "false_literal",
        "literal_bool",
        "literal_boolean",
        "constant_bool",
        "constant_boolean",
    ),
    "Literal_Null": (
        "null",
        "none",
        "nil",
        "nullptr",
        "null_literal",
        "none_literal",
        "nil_literal",
        "literal_null",
        "literal_none",
        "constant_null",
        "constant_none",
    ),
    "Structural_Block": (
        "expr",
        "pass",
        "block",
        "module",
        "compilation_unit",
        "statement_list",
        "expression_list",
        "suite",
    ),
    "Identifier_Context": (
        "load",
        "store",
        "del",
        "starred",
    ),
    "Literal_Generic": (
        "tuple",
        "list",
        "dict",
        "set",
        "operator_array_initializer",
        "literal",
        "constant",
    ),
    "Control_Generic": (
        "try",
        "except_handler",
        "with",
        "withitem",
        "assert",
        "raise",
        "delete",
        "operator_throw",
        "control_structure",
    ),
    "Type_Meta": (
        "alias",
        "method_return",
        "modifier",
        "type",
        "type_ref",
        "annotation",
        "import",
        "import_from",
        "package",
        "namespace_block",
    ),
    "Canonical_Unknown": (
        "canonical_unknown",
        "unknown",
        "unknown_node",
    ),
}


def _build_alias_table() -> dict[str, int]:
    aliases: dict[str, int] = {}
    for category, raw_aliases in _ALIASES_BY_CATEGORY.items():
        category_id = CANONICAL_TYPE_TO_ID[category]
        for raw_alias in (category, *raw_aliases):
            alias = _normalize(raw_alias)
            previous = aliases.setdefault(alias, category_id)
            if previous != category_id:
                raise RuntimeError(f"Conflicting canonical alias: {raw_alias!r}")
    return aliases


RAW_TYPE_TO_CANONICAL_ID: Final[dict[str, int]] = _build_alias_table()


# Namespace-qualified parser classes are common (for example,
# ``com.github.javaparser.ast.stmt.IfStmt``).  Exact aliases cover normal
# inputs; these ordered patterns provide a conservative fallback for qualified
# names and parser-specific prefixes/suffixes.
_PATTERN_RULES: Final[tuple[tuple[int, re.Pattern[str]], ...]] = tuple(
    (CANONICAL_TYPE_TO_ID[category], re.compile(pattern))
    for category, pattern in (
        (
            "Control_If",
            r"(?:^|_)(?:if)(?:_stmt|_statement|_expression|_exp)?(?:_|$)",
        ),
        (
            "Control_Loop",
            r"(?:^|_)(?:for|for_each|foreach|enhanced_for|while|do_while|loop)(?:_stmt|_statement|_expression)?(?:_|$)",
        ),
        (
            "Control_Switch",
            r"(?:^|_)(?:switch|case|match)(?:_stmt|_statement|_expression|_entry|_label)?(?:_|$)",
        ),
        ("Control_Return", r"(?:^|_)(?:return|yield)(?:_stmt|_statement|_expr|_expression)?(?:_|$)"),
        ("Control_Break", r"(?:^|_)(?:break|continue|goto)(?:_stmt|_statement)?(?:_|$)"),
        (
            "Param_Decl",
            r"(?:^|_)(?:formal_|method_)?(?:parameter|param|argument)(?:_in|_out|_declaration|_decl)?(?:_|$)",
        ),
        (
            "Func_Decl",
            r"(?:^|_)(?:function|method|constructor)(?:_declaration|_definition|_def|_decl)(?:_|$)",
        ),
        (
            "Class_Decl",
            r"(?:^|_)(?:class|interface|enum|struct|union|record|type)(?:_declaration|_definition|_specifier|_decl|_def)(?:_|$)",
        ),
        (
            "Var_Decl",
            r"(?:^|_)(?:variable|local_variable|field|member|var|let|const)(?:_declaration|_declarator|_statement|_decl)?(?:_|$)",
        ),
        (
            "Assign_Op",
            r"(?:^|_)(?:assign|assignment|ann_assign|aug_assign|named_expr)(?:_operator|_expression)?(?:_|$)",
        ),
        (
            "Unary_Op",
            r"(?:^|_)(?:unary|prefix|postfix)(?:_op|_operator|_expression)?(?:_|$)",
        ),
        (
            "Binary_Op",
            r"(?:^|_)(?:binary|infix|bin_op|bool_op|compare)(?:_op|_operator|_expression)?(?:_|$)",
        ),
        (
            "Call_Expr",
            r"(?:^|_)(?:call|invocation|method_call|function_call|object_creation|class_instance_creation)(?:_expr|_expression)?(?:_|$)",
        ),
        (
            "Access_Expr",
            r"(?:^|_)(?:identifier|qualified_name|attribute|member_access|field_access|field_expression|array_access|subscript|index_expression)(?:_|$)",
        ),
        (
            "Literal_Bool",
            r"(?:^|_)(?:bool|boolean|true|false)(?:_literal)?(?:_|$)",
        ),
        (
            "Literal_Null",
            r"(?:^|_)(?:null|none|nil|nullptr)(?:_literal)?(?:_|$)",
        ),
        (
            "Literal_Str",
            r"(?:^|_)(?:str|string|char|character|bytes)(?:_literal)?(?:_|$)",
        ),
        (
            "Literal_Num",
            r"(?:^|_)(?:num|number|numeric|integer|int|float|double|decimal|hex|octal|binary_integer)(?:_literal)?(?:_|$)",
        ),
    )
)


def get_canonical_id(raw_type_string: str, *, language: str | None = None) -> int:
    """Return a language-agnostic canonical ID for one parser/Joern node.

    ``language`` is optional to preserve the public mapping API.  It is used
    only for a Joern export quirk: some Java CALL nodes are stored as their
    callee text (for example ``println`` or ``nextLong``) instead of ``CALL``.
    Those labels are safely treated as ``Call_Expr`` after all exact and
    parser-generic rules have failed; Java identifiers themselves are already
    emitted as ``IDENTIFIER`` and are handled before this fallback.
    """
    normalized = _normalize(raw_type_string)
    if not normalized:
        return CANONICAL_UNKNOWN_ID

    exact = RAW_TYPE_TO_CANONICAL_ID.get(normalized)
    if exact is not None:
        return exact

    for category_id, pattern in _PATTERN_RULES:
        if pattern.search(normalized):
            return category_id

    if str(language or "").strip().lower() == "java" and not normalized.startswith("operator_"):
        return CANONICAL_TYPE_TO_ID["Call_Expr"]
    return CANONICAL_UNKNOWN_ID


def get_canonical_name(raw_type_string: str) -> str:
    """Return the canonical category name for a raw node type."""
    return CANONICAL_ID_TO_TYPE[get_canonical_id(raw_type_string)]


def canonicalize_node_types(raw_types: Iterable[str]) -> list[int]:
    """Map an iterable of raw node types while preserving input order."""
    return [get_canonical_id(raw_type) for raw_type in raw_types]


__all__ = [
    "CANONICAL_CATEGORIES",
    "CANONICAL_ID_TO_TYPE",
    "CANONICAL_TYPE_TO_ID",
    "CANONICAL_UNKNOWN_ID",
    "NUM_CANONICAL_TYPES",
    "RAW_TYPE_TO_CANONICAL_ID",
    "canonicalize_node_types",
    "get_canonical_id",
    "get_canonical_name",
]


## Portable clean-data reader and canonical graph conversion

In [ ]:
"""Portable clean-data loading for canonical mono/cross-language experiments.

The module deliberately separates parser-specific raw node strings from the
model-facing representation.  Every node receives a canonical type id and a
small, fixed-size hashed lexical sketch.  Raw strings never become vocabulary
indices, which prevents a Java-only vocabulary from defining the Python/C
feature space during zero-shot evaluation.
"""

from __future__ import annotations

import csv
import gzip
import hashlib
import html
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator, Sequence

import numpy as np
import pandas as pd




RELATIONS = ("ast", "next_token", "ddg")
LEXICAL_PAD_ID = 0
LEXICAL_UNKNOWN_ID = 1
DEFAULT_LEXICAL_BUCKETS = 4096
DEFAULT_LEXICAL_SLOTS = 4


@dataclass(frozen=True)
class PairLanguageFilter:
    """Language constraints for both endpoints of a pair."""

    left: str | None = None
    right: str | None = None

    @classmethod
    def mono(cls, language: str) -> "PairLanguageFilter":
        normalized = normalize_language(language)
        return cls(normalized, normalized)

    @classmethod
    def cross(cls, left: str, right: str) -> "PairLanguageFilter":
        return cls(normalize_language(left), normalize_language(right))


@dataclass(frozen=True)
class CanonicalGraph:
    code_id: str
    language: str
    canonical_ids: np.ndarray
    lexical_ids: np.ndarray
    source_lexical_ids: np.ndarray
    edges: tuple[np.ndarray, np.ndarray, np.ndarray]
    eigenvalues: dict[str, np.ndarray]

    @property
    def node_count(self) -> int:
        return int(len(self.canonical_ids))


def normalize_language(language: str | None) -> str:
    value = (language or "unknown").strip().lower()
    aliases = {
        "javasrc": "java",
        "py": "python",
        "pythonsrc": "python",
        "csrc": "c",
        "c++": "cpp",
        "cxx": "cpp",
    }
    return aliases.get(value, value)


def _is_gzip_file(path: Path) -> bool:
    with path.open("rb") as stream:
        return stream.read(2) == b"\x1f\x8b"


def _open_text(path: Path):
    return gzip.open(path, "rt", encoding="utf-8") if _is_gzip_file(path) else path.open("r", encoding="utf-8")


def _find_file(root: Path, *names: str) -> Path:
    """Find normal gzip/JSON files and Kaggle's occasional ``.tmp`` form.

    Kaggle can expose a ZIP member such as ``codes.jsonl.gz`` as
    ``codes.jsonl/codes.jsonl.gz.tmp``.  The content is still valid gzip, so
    accepting the suffix is safe; ``_open_text`` identifies compression from
    the file magic rather than the filename.
    """
    matches = []
    for name in names:
        variants = (name, f"{name}.tmp")
        for variant in variants:
            direct = root / variant
            if direct.is_file():
                matches.append(direct)
            matches.extend(path for path in root.rglob(variant) if path.is_file())
    if not matches:
        visible = [str(path.relative_to(root)) for path in root.rglob("*") if path.is_file()][:30] if root.exists() else []
        raise FileNotFoundError(f"Missing {names} below {root}. Files seen: {visible}")
    return min(set(matches), key=lambda path: (len(path.parts), str(path)))


def _stable_bucket(value: str, bucket_count: int) -> int:
    if bucket_count <= 2:
        raise ValueError("lexical bucket count must be greater than two")
    digest = hashlib.blake2b(value.encode("utf-8", errors="replace"), digest_size=8, person=b"spectral").digest()
    return 2 + int.from_bytes(digest, "little") % (bucket_count - 2)


_CAMEL_BOUNDARY = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")
_TOKEN_SPLIT = re.compile(r"[^A-Za-z0-9]+")


def lexical_feature_ids(
    raw_label: str,
    *,
    bucket_count: int = DEFAULT_LEXICAL_BUCKETS,
    slots: int = DEFAULT_LEXICAL_SLOTS,
) -> np.ndarray:
    """Return a bounded hashed subtoken/character sketch for a node label.

    Pure line-number labels are intentionally ignored. Numeric, string,
    boolean, and null values are normalized before hashing. The output space
    is fixed across languages and never fitted on the training language.
    """
    text = html.unescape(str(raw_label or "")).strip()
    result = np.full(slots, LEXICAL_PAD_ID, dtype=np.int64)
    if not text or text.isdigit():
        return result
    text = _CAMEL_BOUNDARY.sub(" ", text)
    raw_tokens = [token.lower() for token in _TOKEN_SPLIT.split(text) if token]
    normalized = []
    for token in raw_tokens:
        if token.isdigit():
            token = "<num>"
        elif token in {"true", "false"}:
            token = "<bool>"
        elif token in {"null", "none", "nullptr", "nil"}:
            token = "<null>"
        normalized.append(token)
    # Keep the first few subtokens before character n-grams.  In particular,
    # ``Name,total`` must retain ``total`` even when ``Name`` has many
    # trigrams.  This is a bounded *fallback*, not a train-fitted vocabulary.
    features: list[str] = [f"sub:{token}" for token in normalized]
    for token in normalized[:2]:
        compact = f"^{token}$"
        features.extend(f"tri:{compact[index:index + 3]}" for index in range(max(0, len(compact) - 2)))
    for index, feature in enumerate(features[:slots]):
        result[index] = _stable_bucket(feature, bucket_count)
    return result


_SOURCE_TOKEN = re.compile(r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|&&|\|\||[+*/%<>=!&|~-]")
_SOURCE_KEYWORDS = {
    "def": "func", "function": "func", "void": "func", "class": "class",
    "if": "if", "else": "else", "elif": "else", "switch": "switch", "case": "switch",
    "for": "loop", "while": "loop", "do": "loop", "foreach": "loop",
    "return": "return", "break": "break", "continue": "break",
    "try": "exception", "catch": "exception", "except": "exception", "throw": "exception",
    "import": "import", "from": "import", "package": "import",
    "true": "bool", "false": "bool", "none": "null", "null": "null",
}


def source_lexical_ids(
    source_code: str,
    *,
    bucket_count: int = DEFAULT_LEXICAL_BUCKETS,
    slots: int = 96,
) -> np.ndarray:
    """Return a bounded, language-neutral hashed sketch of source tokens.

    Keywords are normalized to functional concepts and identifiers are split
    into subtokens before hashing.  No train-fitted vocabulary and no raw AST
    node labels are used.  Uniform sampling retains information from the whole
    method instead of favouring Java boilerplate at the beginning.
    """
    output = np.full(slots, LEXICAL_PAD_ID, dtype=np.int64)
    text = html.unescape(str(source_code or ""))
    features = []
    for token in _SOURCE_TOKEN.findall(text):
        normalized = token.lower()
        if normalized.isdigit():
            features.append("src:num")
        elif normalized in _SOURCE_KEYWORDS:
            features.append(f"src:kw:{_SOURCE_KEYWORDS[normalized]}")
        elif normalized[0].isalpha() or normalized[0] == "_":
            split = [part.lower() for part in _CAMEL_BOUNDARY.sub(" ", token).replace("_", " ").split() if part]
            features.extend(f"src:id:{part}" for part in split[:3])
        else:
            features.append(f"src:op:{normalized}")
    if not features:
        return output
    if len(features) <= slots:
        chosen = features
    else:
        chosen = [features[round(index * (len(features) - 1) / (slots - 1))] for index in range(slots)]
    for index, feature in enumerate(chosen[:slots]):
        output[index] = _stable_bucket(feature, bucket_count)
    return output


def lexical_source(raw_type: str, raw_label: str, canonical_id: int) -> str:
    """Select a source-level lexical fallback without using numeric line IDs.

    Python fallback graphs serialize labels such as ``Name,total`` while many
    Joern graphs serialize only a line number.  In the former case the parser
    type prefix is removed; in the latter, an otherwise unknown raw node text
    is used as a weak fallback.  Known structural parser tags are deliberately
    not hashed, because their canonical category already represents them.
    """
    raw_type_text = html.unescape(str(raw_type or "")).strip()
    label = html.unescape(str(raw_label or "")).strip()
    if label:
        prefix = raw_type_text + ","
        if prefix and label.lower().startswith(prefix.lower()):
            label = label[len(prefix):].strip()
        if label and not label.isdigit() and label.lower() != raw_type_text.lower():
            return label
    # The graph exporter loses Joern's original ``CALL`` node kind for labels
    # like ``println``.  Only unmapped entries use that text, and the model
    # gates the hashed feature, so this never becomes a raw-type embedding.
    if canonical_id == get_canonical_id("Canonical_Unknown") and raw_type_text and not raw_type_text.isdigit():
        return raw_type_text
    return ""


def _edge_array(rows: Sequence, cols: Sequence, node_count: int) -> np.ndarray:
    pairs = [
        (int(source), int(target))
        for source, target in zip(rows, cols)
        if 0 <= int(source) < node_count and 0 <= int(target) < node_count
    ]
    return np.asarray(pairs, dtype=np.int32).reshape(-1, 2) if pairs else np.empty((0, 2), dtype=np.int32)


def _project_ddg(ast: dict, ddg: dict, node_count: int) -> np.ndarray:
    ast_ids = [str(value) for value in ast.get("node_ids", [])[:node_count]]
    ast_index = {node_id: index for index, node_id in enumerate(ast_ids)}
    ddg_ids = [str(value) for value in ddg.get("node_ids", [])]
    pairs = []
    for raw_source, raw_target in zip(ddg.get("row", []), ddg.get("col", [])):
        source_position, target_position = int(raw_source), int(raw_target)
        if source_position >= len(ddg_ids) or target_position >= len(ddg_ids):
            continue
        source = ast_index.get(ddg_ids[source_position])
        target = ast_index.get(ddg_ids[target_position])
        if source is not None and target is not None:
            pairs.extend(((source, target), (target, source)))
    return np.asarray(sorted(set(pairs)), dtype=np.int32).reshape(-1, 2) if pairs else np.empty((0, 2), dtype=np.int32)


def canonical_graph_from_record(
    record: dict,
    *,
    language: str,
    max_nodes: int = 512,
    lexical_buckets: int = DEFAULT_LEXICAL_BUCKETS,
    lexical_slots: int = DEFAULT_LEXICAL_SLOTS,
    source_code: str = "",
    source_slots: int = 96,
    strip_node_types: bool = False,
    strip_topology: bool = False,
) -> CanonicalGraph | None:
    graphs = record.get("graphs", {})
    ast_layer = graphs.get("ast", {})
    ddg_layer = graphs.get("ddg", {})
    ast = ast_layer.get("adjacency", ast_layer) if isinstance(ast_layer, dict) else {}
    ddg = ddg_layer.get("adjacency", ddg_layer) if isinstance(ddg_layer, dict) else {}
    raw_types = list(ast.get("node_types", []))
    if not raw_types:
        return None
    node_count = min(len(raw_types), max_nodes)
    raw_labels = list(ast.get("node_labels", []))
    if len(raw_labels) < node_count:
        raw_labels.extend([""] * (node_count - len(raw_labels)))
    # Zero is reserved for padding in the embedding table. Canonical IDs are
    # serialized as 1..N while the mapping API itself remains the requested
    # stable zero-based 0..N-1 vocabulary.
    raw_type_ids = [get_canonical_id(str(raw), language=language) for raw in raw_types[:node_count]]
    canonical_ids = np.asarray([raw_id + 1 for raw_id in raw_type_ids], dtype=np.int64)
    if strip_node_types:
        # One shared id for every node: adjacency survives, categories do not.
        canonical_ids = np.ones_like(canonical_ids)
    lexical_ids = np.stack(
        [
            lexical_feature_ids(
                lexical_source(raw_types[index], raw_labels[index], raw_type_ids[index]),
                bucket_count=lexical_buckets,
                slots=lexical_slots,
            )
            for index in range(node_count)
        ]
    )
    ast_edges = _edge_array(ast.get("row", []), ast.get("col", []), node_count)
    ast_edges = np.asarray(sorted(set(map(tuple, np.vstack((ast_edges, ast_edges[:, ::-1]))))), dtype=np.int32) if len(ast_edges) else ast_edges
    next_edges = np.asarray(
        [(source, target) for index in range(node_count - 1) for source, target in ((index, index + 1), (index + 1, index))],
        dtype=np.int32,
    ).reshape(-1, 2)
    ddg_edges = _project_ddg(ast, ddg, node_count)
    if strip_topology:
        # Input-only ablation: the encoder receives no observed graph edges.
        empty_edges = np.empty((0, 2), dtype=np.int32)
        ast_edges, next_edges, ddg_edges = empty_edges, empty_edges.copy(), empty_edges.copy()
    eigenvalues = {
        graph_type: np.asarray(layer.get("eigenvalues", []), dtype=np.float32)
        for graph_type, layer in graphs.items()
        if isinstance(layer, dict)
    }
    return CanonicalGraph(
        code_id=str(record.get("code_id", record.get("id", ""))),
        language=normalize_language(language),
        canonical_ids=canonical_ids,
        lexical_ids=lexical_ids,
        source_lexical_ids=source_lexical_ids(source_code, bucket_count=lexical_buckets, slots=source_slots),
        edges=(ast_edges, next_edges, ddg_edges),
        eigenvalues=eigenvalues,
    )


class CleanDataCorpus:
    """Reader for ``spectral_clean_data_v1`` exports with language filters."""

    def __init__(self, root: str | Path, *, default_language: str | None = None) -> None:
        self.root = Path(root)
        self.codes_path = _find_file(self.root, "codes.jsonl.gz", "codes.jsonl")
        self.pairs_path = _find_file(self.root, "pairs.csv.gz", "pairs.csv")
        self.graphs_path = _find_file(self.root, "graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp")
        self.default_language = normalize_language(default_language) if default_language else None
        self.code_languages, self.code_texts = self._load_code_metadata()

    def _load_code_metadata(self) -> tuple[dict[str, str], dict[str, str]]:
        languages, code_texts = {}, {}
        with _open_text(self.codes_path) as stream:
            for line in stream:
                if not line.strip():
                    continue
                record = json.loads(line)
                code_id = str(record.get("code_id", record.get("id", record.get("idx", ""))))
                raw_language = record.get("language", record.get("lang", self.default_language))
                if raw_language is None:
                    raise ValueError(
                        f"Code {code_id} in {self.codes_path} has no language; provide default_language explicitly."
                    )
                languages[code_id] = normalize_language(str(raw_language))
                code_texts[code_id] = str(record.get("code", record.get("source", "")) or "")
        return languages, code_texts

    def pairs(
        self,
        split: str,
        *,
        language_filter: PairLanguageFilter | None = None,
        maximum: int | None = None,
        seed: int = 42,
    ) -> pd.DataFrame:
        frame = pd.read_csv(
            self.pairs_path,
            compression="gzip" if _is_gzip_file(self.pairs_path) else None,
            dtype={"left_id": str, "right_id": str, "label": np.int8},
        )
        frame = frame[frame["split"].astype(str).str.lower() == split.strip().lower()].copy()
        frame["left_language"] = frame["left_id"].map(self.code_languages)
        frame["right_language"] = frame["right_id"].map(self.code_languages)
        if frame[["left_language", "right_language"]].isna().any().any():
            raise RuntimeError("At least one pair endpoint is missing from codes.jsonl(.gz).")
        if language_filter:
            if language_filter.left:
                frame = frame[frame["left_language"] == normalize_language(language_filter.left)]
            if language_filter.right:
                frame = frame[frame["right_language"] == normalize_language(language_filter.right)]
        if maximum is not None and len(frame) > maximum:
            pieces = []
            for _, group in frame.groupby("label", sort=True):
                size = max(1, round(maximum * len(group) / len(frame)))
                pieces.append(group.sample(n=min(size, len(group)), random_state=seed))
            frame = pd.concat(pieces).sample(frac=1, random_state=seed).head(maximum)
        return frame.reset_index(drop=True)

    def load_graphs(
        self,
        code_ids: Iterable[str],
        *,
        max_nodes: int = 512,
        lexical_buckets: int = DEFAULT_LEXICAL_BUCKETS,
        lexical_slots: int = DEFAULT_LEXICAL_SLOTS,
        source_slots: int = 96,
        strip_node_types: bool = False,
        strip_topology: bool = False,
    ) -> dict[str, CanonicalGraph]:
        required = {str(value) for value in code_ids}
        graphs = {}
        with _open_text(self.graphs_path) as stream:
            for line in stream:
                record = json.loads(line)
                code_id = str(record.get("code_id", record.get("id", "")))
                if code_id not in required:
                    continue
                graph = canonical_graph_from_record(
                    record,
                    language=self.code_languages[code_id],
                    max_nodes=max_nodes,
                    lexical_buckets=lexical_buckets,
                    lexical_slots=lexical_slots,
                    source_code=self.code_texts.get(code_id, ""),
                    source_slots=source_slots,
                    strip_node_types=strip_node_types,
                    strip_topology=strip_topology,
                )
                if graph is not None:
                    graphs[code_id] = graph
        return graphs


def pair_code_ids(*frames: pd.DataFrame) -> set[str]:
    values: set[str] = set()
    for frame in frames:
        values.update(frame["left_id"].astype(str))
        values.update(frame["right_id"].astype(str))
    return values


## PyTorch graph batching

In [ ]:
"""PyTorch Dataset/DataLoader adapters for canonical code graphs."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Sequence

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset




@dataclass
class CanonicalGraphBatch:
    canonical_ids: torch.Tensor
    lexical_ids: torch.Tensor
    source_lexical_ids: torch.Tensor
    mask: torch.Tensor
    edges: torch.Tensor

    def to(self, device: str | torch.device) -> "CanonicalGraphBatch":
        return CanonicalGraphBatch(
            self.canonical_ids.to(device, non_blocking=True),
            self.lexical_ids.to(device, non_blocking=True),
            self.source_lexical_ids.to(device, non_blocking=True),
            self.mask.to(device, non_blocking=True),
            self.edges.to(device, non_blocking=True),
        )


class CanonicalPairDataset(Dataset):
    def __init__(self, pairs: pd.DataFrame, graphs: dict[str, CanonicalGraph]) -> None:
        usable = pairs[pairs["left_id"].isin(graphs) & pairs["right_id"].isin(graphs)].copy()
        self.dropped_pairs = int(len(pairs) - len(usable))
        if usable.empty:
            raise RuntimeError("No usable pairs remain after joining pair endpoints to graph records.")
        self.left_ids = usable["left_id"].astype(str).tolist()
        self.right_ids = usable["right_id"].astype(str).tolist()
        self.labels = usable["label"].to_numpy(dtype=np.float32)
        self.graphs = graphs

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int):
        return self.graphs[self.left_ids[index]], self.graphs[self.right_ids[index]], self.labels[index]


def pack_graphs(graphs: Sequence[CanonicalGraph], max_nodes: int) -> CanonicalGraphBatch:
    batch_size = len(graphs)
    lexical_slots = graphs[0].lexical_ids.shape[1]
    canonical_ids = torch.zeros(batch_size, max_nodes, dtype=torch.long)
    lexical_ids = torch.zeros(batch_size, max_nodes, lexical_slots, dtype=torch.long)
    source_slots = graphs[0].source_lexical_ids.shape[0]
    source_lexical = torch.zeros(batch_size, source_slots, dtype=torch.long)
    mask = torch.zeros(batch_size, max_nodes, dtype=torch.bool)
    edges = torch.zeros(batch_size, 3, max_nodes, max_nodes, dtype=torch.float32)
    for batch_index, graph in enumerate(graphs):
        node_count = min(graph.node_count, max_nodes)
        canonical_ids[batch_index, :node_count] = torch.from_numpy(graph.canonical_ids[:node_count])
        lexical_ids[batch_index, :node_count] = torch.from_numpy(graph.lexical_ids[:node_count])
        source_lexical[batch_index] = torch.from_numpy(graph.source_lexical_ids)
        mask[batch_index, :node_count] = True
        for relation, relation_edges in enumerate(graph.edges):
            if not len(relation_edges):
                continue
            valid = relation_edges[(relation_edges[:, 0] < node_count) & (relation_edges[:, 1] < node_count)]
            if len(valid):
                rows = torch.from_numpy(valid[:, 0].astype(np.int64))
                cols = torch.from_numpy(valid[:, 1].astype(np.int64))
                edges[batch_index, relation, rows, cols] = 1.0
    return CanonicalGraphBatch(canonical_ids, lexical_ids, source_lexical, mask, edges)


def make_collate(max_nodes: int):
    def collate(items):
        left, right, labels = zip(*items)
        return pack_graphs(left, max_nodes), pack_graphs(right, max_nodes), torch.tensor(labels, dtype=torch.float32)

    return collate


def make_loader(
    pairs: pd.DataFrame,
    graphs: dict[str, CanonicalGraph],
    *,
    max_nodes: int,
    batch_size: int,
    shuffle: bool,
    num_workers: int = 0,
    pin_memory: bool = True,
) -> DataLoader:
    dataset = CanonicalPairDataset(pairs, graphs)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=make_collate(max_nodes),
    )


## Spectral descriptor-only pair model

The encoder may use canonical, lexical, and topology inputs to construct the learned latent graph and its spectral signals. The symmetric classifier never receives those inputs or code embeddings directly. It compares density, heat trace, and (when enabled) Chebyshev energy as separate normalized blocks using symmetric differences, products, log-distances, and per-block summaries.


In [ ]:
"""Canonical SPECTRA-Siam model for mono- and zero-shot cross-language runs.

The encoder consumes only stable canonical category IDs plus a bounded hashed
lexical sketch. Parser-specific raw node strings never index an embedding
table. Input topology is retained as a soft latent-graph prior.
"""

from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F




@dataclass(frozen=True)
class CanonicalSpectraConfig:
    canonical_vocab_size: int = NUM_CANONICAL_TYPES + 1  # zero is padding
    lexical_buckets: int = 4096
    hidden_dim: int = 256
    latent_nodes: int = 32
    slot_iterations: int = 3
    attention_heads: int = 4
    lexical_dropout: float = 0.30
    use_node_lexical: bool = True
    use_source_lexical: bool = False
    readout_mode: str = "graph_signal_spectral"
    target_density: float = 0.15
    chebyshev_degree: int = 12
    density_bins: int = 32
    heat_samples: int = 24
    spectral_bands: int = 12
    spectral_signals: int = 8
    dropout: float = 0.10
    relation_indices: tuple[int, ...] = (0,)  # AST only until cross-layer node alignment is guaranteed

    @property
    def descriptor_dim(self) -> int:
        return self.density_bins + self.heat_samples + self.spectral_bands * self.spectral_signals

    @property
    def eigenvalue_descriptor_dim(self) -> int:
        return self.density_bins + self.heat_samples

    @property
    def readout_descriptor_dim(self) -> int:
        if self.readout_mode == "eigenvalue_only":
            return self.eigenvalue_descriptor_dim
        return self.descriptor_dim


class RelationGraphLayer(nn.Module):
    def __init__(self, dim: int, relations: int = 3, dropout: float = 0.10) -> None:
        super().__init__()
        self.self_linear = nn.Linear(dim, dim)
        self.relation_linears = nn.ModuleList(nn.Linear(dim, dim, bias=False) for _ in range(relations))
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, states: torch.Tensor, edges: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        message = self.self_linear(states)
        for relation, linear in enumerate(self.relation_linears):
            adjacency = edges[:, relation]
            degree = adjacency.sum(-1, keepdim=True).clamp_min(1.0)
            message = message + torch.bmm(adjacency / degree, linear(states))
        return self.norm(states + self.dropout(F.gelu(message))) * mask.unsqueeze(-1)


class SlotAttentionInduction(nn.Module):
    def __init__(self, dim: int, slots: int, iterations: int) -> None:
        super().__init__()
        self.slots = slots
        self.iterations = iterations
        self.scale = dim**-0.5
        self.slot_seed = nn.Parameter(torch.randn(1, slots, dim) * 0.02)
        self.input_norm = nn.LayerNorm(dim)
        self.slot_norm = nn.LayerNorm(dim)
        self.to_query = nn.Linear(dim, dim, bias=False)
        self.to_key = nn.Linear(dim, dim, bias=False)
        self.to_value = nn.Linear(dim, dim, bias=False)
        self.update = nn.GRUCell(dim, dim)
        self.mlp = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))

    def forward(self, states: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        normalized = self.input_norm(states)
        key, value = self.to_key(normalized), self.to_value(normalized)
        slots = self.slot_seed.expand(states.size(0), -1, -1)
        for _ in range(self.iterations):
            scores = torch.einsum("bsd,bnd->bns", self.to_query(self.slot_norm(slots)), key) * self.scale
            scores = scores.masked_fill(~mask.unsqueeze(-1), -1e4)
            assignments = torch.softmax(scores, dim=-1)
            weights = assignments / assignments.sum(1, keepdim=True).clamp_min(1e-6)
            updates = torch.einsum("bns,bnd->bsd", weights, value)
            slots = self.update(updates.reshape(-1, updates.size(-1)), slots.reshape(-1, slots.size(-1))).view_as(slots)
            slots = slots + self.mlp(slots)
        return slots, assignments


class LatentGraphRefinement(nn.Module):
    def __init__(self, dim: int, dropout: float) -> None:
        super().__init__()
        self.message = nn.Linear(dim, dim, bias=False)
        self.self_linear = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, states: torch.Tensor, adjacency: torch.Tensor) -> torch.Tensor:
        degree = adjacency.sum(-1, keepdim=True).clamp_min(1e-6)
        message = torch.bmm(adjacency / degree, self.message(states))
        return self.norm(states + self.dropout(F.gelu(self.self_linear(states) + message)))


class MultiScaleSpectralExtractor(nn.Module):
    """Graph spectrum plus attributed graph-signal spectral energies.

    In ``graph_signal_spectral`` mode, canonical/lexical information can reach
    the classifier only after node signals have been filtered by polynomial
    functions of the learned latent Laplacian.  Squared energy is aggregated
    over nodes, making the descriptor invariant to latent-node permutations.
    There is no state pooling or raw-feature concatenation in this branch.
    """

    def __init__(self, dim: int, config: CanonicalSpectraConfig) -> None:
        super().__init__()
        self.config = config
        if config.readout_mode in {"hybrid", "graph_signal_spectral"}:
            self.signal = nn.Linear(dim, config.spectral_signals, bias=False)
        else:
            self.signal = None
        self.band_attention = (
            nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, config.spectral_bands))
            if config.readout_mode == "hybrid"
            else None
        )

        grid = torch.linspace(0, 2, 256)
        theta = torch.acos((grid - 1).clamp(-1, 1))
        coefficients = []
        for center in torch.linspace(0.05, 1.95, config.spectral_bands):
            response = torch.exp(-0.5 * ((grid - center) / 0.18) ** 2)
            coefficients.append(
                torch.stack(
                    [
                        (response * torch.cos(order * theta)).mean() * (1 if order == 0 else 2)
                        for order in range(config.chebyshev_degree + 1)
                    ]
                )
            )
        self.register_buffer("chebyshev_coefficients", torch.stack(coefficients))
        self.register_buffer("density_centers", torch.linspace(0, 2, config.density_bins))
        self.register_buffer("heat_times", torch.logspace(-2, 2, config.heat_samples))

    def forward(self, states: torch.Tensor, adjacency: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Eigensolver and graph filtering remain float32 under AMP.
        with torch.autocast(device_type=states.device.type, enabled=False):
            states, adjacency = states.float(), adjacency.float()
            batch_size, node_count, _ = adjacency.shape
            identity = torch.eye(node_count, device=adjacency.device).unsqueeze(0)
            adjacency = torch.nan_to_num(adjacency, nan=0.0, posinf=1.0, neginf=0.0).clamp(0.0, 1.0)
            adjacency = 0.5 * (adjacency + adjacency.transpose(1, 2)) * (1 - identity)
            degree = adjacency.sum(-1).clamp_min(1e-6)
            normalized = adjacency * degree.rsqrt().unsqueeze(-1) * degree.rsqrt().unsqueeze(-2)
            laplacian = identity - normalized
            laplacian = 0.5 * (laplacian + laplacian.transpose(1, 2))

            if self.config.readout_mode in {"eigenvalue_only", "graph_signal_spectral"}:
                eigenvalues = torch.linalg.eigvalsh(laplacian).clamp(0, 2)
            else:
                with torch.no_grad():
                    eigenvalues = torch.linalg.eigvalsh(laplacian).clamp(0, 2)
            density = torch.exp(
                -0.5 * ((eigenvalues.unsqueeze(-1) - self.density_centers) / 0.08) ** 2
            ).mean(1)
            heat_trace = torch.exp(-eigenvalues.unsqueeze(-1) * self.heat_times).mean(1)

            if self.config.readout_mode == "eigenvalue_only":
                descriptor = torch.cat((density, heat_trace), dim=-1)
                band_weights = torch.full(
                    (batch_size, self.config.spectral_bands),
                    1.0 / self.config.spectral_bands,
                    device=adjacency.device,
                )
            else:
                if self.signal is None:
                    raise RuntimeError("Graph-signal projection was not initialized")
                # A learned channel projection creates a compact attributed
                # graph signal. RMS normalization prevents raw signal scale
                # from acting as a non-spectral shortcut.
                signal = self.signal(states)
                signal = signal / signal.square().mean(1, keepdim=True).add(1e-6).sqrt()

                # T_k(L-I)X is a Chebyshev approximation of spectral filters
                # g_b(L)X. No node state skips these graph-dependent operators.
                shifted = laplacian - identity
                terms = [signal, torch.bmm(shifted, signal)]
                for _ in range(2, self.config.chebyshev_degree + 1):
                    terms.append(2 * torch.bmm(shifted, terms[-1]) - terms[-2])
                filtered = torch.einsum(
                    "bkse,ck->bcse", torch.stack(terms, dim=1), self.chebyshev_coefficients
                )
                raw_energy = torch.log1p(filtered.square().mean(2))

                if self.config.readout_mode == "graph_signal_spectral":
                    # Fixed bands prevent an input-conditioned state gate from
                    # becoming a covert lexical/node-embedding bypass.
                    band_weights = torch.full(
                        (batch_size, self.config.spectral_bands),
                        1.0 / self.config.spectral_bands,
                        device=adjacency.device,
                    )
                    energy = raw_energy.reshape(batch_size, -1)
                else:
                    if self.band_attention is None:
                        raise RuntimeError("Hybrid spectral attention was not initialized")
                    band_weights = (
                        torch.softmax(self.band_attention(states.mean(1)), dim=-1)
                        if USE_SPECTRAL_ATTENTION
                        else torch.full(
                            (batch_size, self.config.spectral_bands),
                            1.0 / self.config.spectral_bands,
                            device=adjacency.device,
                        )
                    )
                    energy = (raw_energy * band_weights.unsqueeze(-1)).reshape(batch_size, -1)
                descriptor = torch.cat((density, heat_trace, energy), dim=-1)
        return descriptor, eigenvalues, band_weights


class CanonicalSpectraEncoder(nn.Module):
    def __init__(self, config: CanonicalSpectraConfig) -> None:
        super().__init__()
        if config.hidden_dim % config.attention_heads:
            raise ValueError("hidden_dim must be divisible by attention_heads")
        if config.readout_mode not in {"hybrid", "eigenvalue_only", "graph_signal_spectral"}:
            raise ValueError("unsupported readout_mode")
        if config.readout_mode != "hybrid" and config.use_source_lexical:
            raise ValueError("source lexical residual is incompatible with spectral-only readouts")
        self.config = config
        dim = config.hidden_dim
        self.canonical_embedding = nn.Embedding(config.canonical_vocab_size, dim, padding_idx=0)
        self.lexical_embedding = nn.Embedding(config.lexical_buckets, dim, padding_idx=0)
        # Start lexical fallback at roughly 20% of the canonical signal.  It
        # can help when an identifier/literal is available, but cannot become
        # a direct raw-node-type lookup table.
        self.lexical_gate_logit = nn.Parameter(torch.tensor(-1.38629436))
        self.lexical_dropout = nn.Dropout(config.lexical_dropout)
        if config.use_source_lexical:
            self.source_projection = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.LayerNorm(dim))
            self.source_gate_logit = nn.Parameter(torch.tensor(-0.61903921))
        else:
            self.source_projection = None
            self.register_parameter("source_gate_logit", None)
        self.input_norm = nn.LayerNorm(dim)
        if not config.relation_indices:
            raise ValueError("At least one input relation must be enabled.")
        self.graph_layers = nn.ModuleList(
            [RelationGraphLayer(dim, relations=len(config.relation_indices), dropout=config.dropout) for _ in range(2)]
        )
        self.slot_attention = SlotAttentionInduction(dim, config.latent_nodes, config.slot_iterations)
        self.adjacency_query = nn.Linear(dim, dim, bias=False)
        self.adjacency_key = nn.Linear(dim, dim, bias=False)
        self.prior_scale = nn.Parameter(torch.tensor(1.0))
        self.temperature_bias = nn.Parameter(torch.tensor(-0.55))
        self.temperature_complexity = nn.Parameter(torch.tensor(1.25))
        self.refinement = nn.ModuleList([LatentGraphRefinement(dim, config.dropout) for _ in range(2)])
        self.spectral = MultiScaleSpectralExtractor(dim, config)
        self.spectral_projection = nn.Sequential(
            nn.LayerNorm(config.readout_descriptor_dim), nn.Linear(config.readout_descriptor_dim, dim), nn.GELU()
        )
        self.output = (
            nn.Sequential(nn.LayerNorm(dim * 2), nn.Linear(dim * 2, dim), nn.GELU(), nn.LayerNorm(dim))
            if config.readout_mode == "hybrid"
            else None
        )

    def forward(self, batch, temperature: float = 0.30):
        canonical = self.canonical_embedding(batch.canonical_ids.long())
        if self.config.use_node_lexical:
            lexical = self.lexical_embedding(batch.lexical_ids.long()).mean(-2)
            lexical = self.lexical_dropout(lexical) if self.training else lexical
            canonical = canonical + torch.sigmoid(self.lexical_gate_logit) * lexical
        input_features = self.input_norm(canonical) * batch.mask.unsqueeze(-1)
        enabled_edges = batch.edges[:, self.config.relation_indices]
        states = input_features
        for layer in self.graph_layers:
            states = layer(states, enabled_edges, batch.mask)

        slots, assignment = self.slot_attention(states, batch.mask)
        assignment = torch.nan_to_num(assignment)
        assignment = assignment / assignment.sum(-1, keepdim=True).clamp_min(1e-6)
        reconstruction = torch.bmm(assignment, slots)
        reconstruction_loss = (
            (input_features - reconstruction).square().mean(-1) * batch.mask
        ).sum() / batch.mask.sum().clamp_min(1)

        input_adjacency = enabled_edges.sum(1).clamp(max=1)
        structural_prior = torch.bmm(
            assignment.transpose(1, 2), torch.bmm(input_adjacency, assignment)
        )
        structural_prior = structural_prior / structural_prior.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6)

        head_dim = self.config.hidden_dim // self.config.attention_heads
        with torch.autocast(device_type=slots.device.type, enabled=False):
            query = self.adjacency_query(slots.float()).view(
                -1, self.config.latent_nodes, self.config.attention_heads, head_dim
            ).transpose(1, 2)
            key = self.adjacency_key(slots.float()).view(
                -1, self.config.latent_nodes, self.config.attention_heads, head_dim
            ).transpose(1, 2)
            score = torch.matmul(query, key.transpose(-1, -2)).mean(1) / math.sqrt(head_dim)
            score = 0.5 * (score + score.transpose(1, 2)) + self.prior_scale.float() * structural_prior.float()
            score = torch.nan_to_num(score, nan=0.0, posinf=20.0, neginf=-20.0).clamp(-20, 20)
            identity = torch.eye(self.config.latent_nodes, device=score.device).unsqueeze(0)
            node_complexity = batch.mask.float().mean(-1)
            edge_complexity = enabled_edges.float().sum(dim=(1, 2, 3)) / batch.mask.float().sum(-1).square().clamp_min(1.0)
            complexity = (0.75 * node_complexity + 0.25 * edge_complexity.clamp(0, 1)).clamp(0, 1)
            adaptive_temperature = ADAPTIVE_TEMPERATURE_MIN + (ADAPTIVE_TEMPERATURE_MAX - ADAPTIVE_TEMPERATURE_MIN) * torch.sigmoid(self.temperature_bias.float() + F.softplus(self.temperature_complexity.float()) * complexity) if USE_ADAPTIVE_TEMPERATURE else torch.full_like(complexity, TEMPERATURE_END)
            # Keep a differentiable weighted graph. A fixed top-k gave
            # every example nearly the same edge count and erased precisely the
            # graph-to-graph spectral variation that the readout needs.
            adjacency = torch.sigmoid(score / adaptive_temperature[:, None, None]) * (1 - identity)
            adjacency = 0.5 * (adjacency + adjacency.transpose(1, 2))

        projected_topology = torch.bmm(assignment, torch.bmm(adjacency.to(assignment.dtype), assignment.transpose(1, 2)))
        projected_topology = projected_topology.clamp(1e-6, 1 - 1e-6)
        topology_mask = batch.mask.unsqueeze(1) & batch.mask.unsqueeze(2)
        topology_mask = topology_mask & ~torch.eye(
            topology_mask.size(-1), device=topology_mask.device, dtype=torch.bool
        ).unsqueeze(0)
        topology_target = torch.maximum(input_adjacency, input_adjacency.transpose(1, 2)).to(projected_topology.dtype)
        positive_edges = (topology_target * topology_mask).sum()
        negative_edges = ((1 - topology_target) * topology_mask).sum()
        topology_positive_weight = (negative_edges / positive_edges.clamp_min(1)).detach().clamp(1, 20)
        topology_element = -(
            topology_positive_weight * topology_target * projected_topology.log()
            + (1 - topology_target) * (1 - projected_topology).log()
        )
        topology_reconstruction_loss = (
            topology_element * topology_mask
        ).sum() / topology_mask.sum().clamp_min(1)

        latent = slots
        for layer in self.refinement:
            latent = layer(latent, adjacency.to(latent.dtype))
        descriptor, eigenvalues, band_weights = self.spectral(latent, adjacency)
        spectral_embedding = self.spectral_projection(descriptor).to(latent.dtype)
        if self.config.readout_mode in {"eigenvalue_only", "graph_signal_spectral"}:
            # In graph_signal_spectral mode this embedding is derived only from
            # g_b(L)X energies and adjacency-spectrum summaries.
            embedding = F.normalize(spectral_embedding, dim=-1)
        else:
            if self.output is None:
                raise RuntimeError("Hybrid latent fusion module was not initialized")
            embedding = F.normalize(
                self.output(torch.cat((latent.mean(1), spectral_embedding), dim=-1)), dim=-1
            )
        if self.config.use_source_lexical:
            if self.source_projection is None or self.source_gate_logit is None:
                raise RuntimeError("Source lexical residual was not initialized")
            source = self.lexical_embedding(batch.source_lexical_ids.long()).mean(1)
            source = self.source_projection(source)
            embedding = F.normalize(embedding + torch.sigmoid(self.source_gate_logit) * source, dim=-1)
        density = adjacency.sum(dim=(1, 2)) / (
            self.config.latent_nodes * (self.config.latent_nodes - 1)
        )
        density_loss = (density - self.config.target_density).square().mean()
        connectivity_loss = F.relu(0.03 - eigenvalues[:, 1]).square().mean()
        return embedding, descriptor, {
            "reconstruction": reconstruction_loss,
            "topology_reconstruction": topology_reconstruction_loss,
            "density": density_loss,
            "connectivity": connectivity_loss,
            "adaptive_temperature": adaptive_temperature,
            "spectral_attention_entropy": -(band_weights * band_weights.clamp_min(1e-8).log()).sum(-1),
            "graph": 2 * density_loss + 0.1 * connectivity_loss,
        }


class CanonicalSpectraSiam(nn.Module):
    """Symmetric comparator over separately normalized spectral blocks only."""

    def __init__(self, config: CanonicalSpectraConfig) -> None:
        super().__init__()
        self.config = config
        self.encoder = CanonicalSpectraEncoder(config)
        # Density and heat trace are always separate comparison blocks.  The
        # attributed Chebyshev-energy block exists only in spectral-signal modes.
        self.block_sizes = (config.density_bins, config.heat_samples)
        if config.readout_mode != "eigenvalue_only":
            self.block_sizes += (config.spectral_bands * config.spectral_signals,)
        pair_dim = 3 * config.readout_descriptor_dim + 3 * len(self.block_sizes)
        pair_hidden_dim = config.hidden_dim + config.hidden_dim // 2
        pair_bottleneck_dim = config.hidden_dim - config.hidden_dim // 8
        self.classifier = nn.Sequential(
            nn.LayerNorm(pair_dim),
            nn.Linear(pair_dim, pair_hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(pair_hidden_dim, pair_bottleneck_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(pair_bottleneck_dim, 1),
        )

    def _spectral_pair_features(
        self, left_spectrum: torch.Tensor, right_spectrum: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        left_blocks = torch.split(left_spectrum, self.block_sizes, dim=-1)
        right_blocks = torch.split(right_spectrum, self.block_sizes, dim=-1)
        interactions = []
        summaries = []
        block_cosines = []
        for left_block, right_block in zip(left_blocks, right_blocks):
            left_log = torch.log1p(left_block.clamp_min(0))
            right_log = torch.log1p(right_block.clamp_min(0))
            left_shape = F.normalize(left_log, p=2, dim=-1)
            right_shape = F.normalize(right_log, p=2, dim=-1)
            difference = (left_shape - right_shape).abs()
            interactions.extend((difference, left_shape * right_shape, (left_log - right_log).abs()))
            block_cosine = F.cosine_similarity(left_shape, right_shape)
            block_cosines.append(block_cosine)
            summaries.extend(
                (
                    block_cosine,
                    difference.mean(-1),
                    difference.square().mean(-1).add(1e-8).sqrt(),
                )
            )
        # The head sees only symmetric, block-wise spectral comparisons.  It
        # never receives an individual descriptor, code embedding, pooled node
        # state, lexical vector, or source-token feature.
        pair_features = torch.cat((*interactions, torch.stack(summaries, dim=-1)), dim=-1)
        spectral_cosine = torch.stack(block_cosines, dim=-1).mean(-1)
        return pair_features, spectral_cosine

    def forward(self, left, right, temperature: float = 0.30):
        left_embedding, left_spectrum, left_aux = self.encoder(left, temperature)
        right_embedding, right_spectrum, right_aux = self.encoder(right, temperature)
        spectral_pair_features, spectral_cosine = self._spectral_pair_features(
            left_spectrum, right_spectrum
        )
        auxiliary = {key: 0.5 * (left_aux[key] + right_aux[key]) for key in left_aux}
        auxiliary["spectral_cosine"] = spectral_cosine
        # Embedding similarity is diagnostic metadata only.  It is excluded
        # from the classifier and every non-zero training-loss term.
        auxiliary["embedding_cosine"] = F.cosine_similarity(left_embedding, right_embedding)
        spectral_batch = torch.cat(
            (F.normalize(left_spectrum, dim=-1), F.normalize(right_spectrum, dim=-1)), dim=0
        )
        spectral_std = torch.sqrt(spectral_batch.var(dim=0, unbiased=False) + 1e-4).mean()
        auxiliary["spectral_variance"] = F.relu(SPECTRAL_VARIANCE_FLOOR - spectral_std)
        return self.classifier(spectral_pair_features).squeeze(-1), auxiliary

def canonical_spectra_loss(
    logits: torch.Tensor,
    auxiliary: dict[str, torch.Tensor],
    labels: torch.Tensor,
    *,
    positive_class_weight: float = 1.0,
    spectral_weight: float = 0.30,
    auc_ranking_weight: float = 0.10,
    reconstruction_weight: float = 0.05,
    topology_reconstruction_weight: float = 0.05,
    spectral_variance_weight: float = 0.05,
    graph_weight: float = 0.01,
    embedding_contrastive_weight: float = 0.0,
    embedding_negative_margin: float = 0.20,
    hard_negative_weight: float = 0.20,
    hard_negative_margin: float = 0.10,
    hard_negative_fraction: float = 0.25,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    labels = labels.float()
    if positive_class_weight <= 0:
        raise ValueError("positive_class_weight must be positive")
    classification = F.binary_cross_entropy_with_logits(
        logits,
        labels,
        pos_weight=torch.as_tensor(positive_class_weight, device=logits.device, dtype=logits.dtype),
    )
    positive = labels * (1 - auxiliary["spectral_cosine"]).square()
    negative = (1 - labels) * F.relu(auxiliary["spectral_cosine"] - 0.25).square()
    spectral_contrastive = (positive + negative).mean()
    positive_logits = logits[labels > 0.5]
    negative_logits = logits[labels < 0.5]
    if positive_logits.numel() and negative_logits.numel():
        auc_ranking = F.softplus(
            -(positive_logits.unsqueeze(1) - negative_logits.unsqueeze(0))
        ).mean()
    else:
        auc_ranking = logits.new_zeros(())

    if embedding_contrastive_weight != 0:
        raise ValueError("embedding_contrastive_weight must remain zero for descriptor-only SPECTRA-Siam")
    embedding_contrastive = logits.new_zeros(())

    # Hard-negative mining is based on the block-wise spectral comparison,
    # never on a standalone embedding similarity.
    known_negative_scores = F.relu(
        auxiliary["spectral_cosine"][labels < 0.5] - hard_negative_margin
    ).square()
    if known_negative_scores.numel():
        hard_count = max(1, int(known_negative_scores.numel() * hard_negative_fraction))
        hard_negative = torch.topk(known_negative_scores, hard_count).values.mean()
    else:
        hard_negative = logits.new_zeros(())


    total = (
        classification
        + spectral_weight * spectral_contrastive
        + auc_ranking_weight * auc_ranking
        + hard_negative_weight * hard_negative
        + reconstruction_weight * auxiliary["reconstruction"]
        + topology_reconstruction_weight * auxiliary["topology_reconstruction"]
        + spectral_variance_weight * auxiliary["spectral_variance"]
        + graph_weight * auxiliary["graph"]
    )
    parts = {
        "classification": classification.detach(),
        "spectral_contrastive": spectral_contrastive.detach(),
        "auc_ranking": auc_ranking.detach(),
        "embedding_contrastive": embedding_contrastive.detach(),
        "hard_negative": hard_negative.detach(),
        "reconstruction": auxiliary["reconstruction"].detach(),
        "topology_reconstruction": auxiliary["topology_reconstruction"].detach(),
        "spectral_variance": auxiliary["spectral_variance"].detach(),
        "graph": auxiliary["graph"].detach(),
    }
    return total, parts


## Training and evaluation

The run uses every language available in the attached V3 dataset and samples a stratified, reproducible subset.


In [ ]:
# Canonical multi-language SPECTRA-Siam training/evaluation helpers.
DIAGNOSTIC_REPORTS: dict[str, dict] = {}

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def _summary(values: np.ndarray) -> dict:
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        return {"count": 0}
    return {
        "count": int(len(values)),
        "mean": float(values.mean()),
        "std": float(values.std()),
        "q01": float(np.quantile(values, .01)),
        "q05": float(np.quantile(values, .05)),
        "q50": float(np.quantile(values, .50)),
        "q95": float(np.quantile(values, .95)),
        "q99": float(np.quantile(values, .99)),
    }


def _js_divergence(left: np.ndarray, right: np.ndarray) -> float:
    left = np.asarray(left, dtype=np.float64); right = np.asarray(right, dtype=np.float64)
    left /= left.sum() or 1.0; right /= right.sum() or 1.0; midpoint = .5 * (left + right)
    def kl(value):
        mask = value > 0
        return float((value[mask] * np.log2(value[mask] / midpoint[mask])).sum())
    return .5 * (kl(left) + kl(right))


def report_input_diagnostics(frames: dict[str, pd.DataFrame], graphs: dict) -> dict:
    """Check canonical-space alignment and pair-language composition before training."""
    by_language = {}
    for language in sorted({graph.language for graph in graphs.values()}):
        ids = [graph.canonical_ids for graph in graphs.values() if graph.language == language]
        values = np.concatenate(ids) if ids else np.empty(0, dtype=np.int64)
        counts = np.bincount(values, minlength=NUM_CANONICAL_TYPES + 1)[1:]
        by_language[language] = {
            "graphs": int(len(ids)), "nodes": int(values.size),
            "mean_nodes": float(values.size / max(1, len(ids))),
            "unknown_rate": float((values == CANONICAL_UNKNOWN_ID + 1).mean()) if values.size else 0.0,
            "canonical_counts": {CANONICAL_ID_TO_TYPE[i]: int(counts[i]) for i in range(NUM_CANONICAL_TYPES) if counts[i]},
        }
    languages = sorted(by_language)
    alignment = {}
    if len(languages) >= 2:
        for index, left in enumerate(languages):
            for right in languages[index + 1:]:
                left_counts = np.asarray([by_language[left]["canonical_counts"].get(name, 0) for name in CANONICAL_ID_TO_TYPE])
                right_counts = np.asarray([by_language[right]["canonical_counts"].get(name, 0) for name in CANONICAL_ID_TO_TYPE])
                alignment[f"{left}__{right}_jsd_bits"] = _js_divergence(left_counts, right_counts)
    pair_composition = {}
    for split, frame in frames.items():
        if {"left_language", "right_language", "label"}.issubset(frame.columns):
            pair_composition[split] = {
                f"{left}-{right}-label{label}": int(count)
                for (left, right, label), count in frame.groupby(["left_language", "right_language", "label"]).size().items()
            }
    report = {"canonical_by_language": by_language, "alignment": alignment, "pair_composition": pair_composition}
    print("\n=== Canonical mapping diagnostic ===")
    for language, value in by_language.items():
        print(f"{language}: graphs={value['graphs']:,}, mean_nodes={value['mean_nodes']:.1f}, unknown={value['unknown_rate']:.2%}")
    print("alignment (JSD; lower means closer):", alignment)
    print("pair languages:", pair_composition)
    (WORK_DIR / f"spectra_siam_{RUN_TAG}_mapping_diagnostic.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


def report_prediction_diagnostics(model, loader, temperature: float, split: str) -> dict:
    """Expose logit/probability/spectral-similarity collapse during diagnostic runs."""
    model.eval(); logits_all, labels_all, spectra_all, embeddings_all = [], [], [], []
    with torch.no_grad():
        for left, right, labels in tqdm(loader, desc=f"{split} diagnostic", leave=False):
            logits, auxiliary = model(left.to(DEVICE), right.to(DEVICE), temperature)
            logits_all.append(logits.float().cpu().numpy())
            labels_all.append(labels.numpy())
            spectra_all.append(auxiliary["spectral_cosine"].float().cpu().numpy())
            embeddings_all.append(auxiliary["embedding_cosine"].float().cpu().numpy())
    logits = np.concatenate(logits_all); labels = np.concatenate(labels_all).astype(np.int64)
    probabilities = 1.0 / (1.0 + np.exp(-np.clip(logits, -60, 60)))
    spectra = np.concatenate(spectra_all); embeddings = np.concatenate(embeddings_all)
    report = {
        "split": split,
        "logits": _summary(logits), "probabilities": _summary(probabilities),
        "spectral_cosine": _summary(spectra), "embedding_cosine": _summary(embeddings),
        "probability_ge_0_5": float((probabilities >= .5).mean()),
        "metrics_at_0_5": metric_summary(labels, probabilities, .5),
        "probability_by_label": {str(label): _summary(probabilities[labels == label]) for label in (0, 1)},
        "spectral_cosine_by_label": {str(label): _summary(spectra[labels == label]) for label in (0, 1)},
        "embedding_cosine_by_label": {str(label): _summary(embeddings[labels == label]) for label in (0, 1)},
    }
    print(f"\n=== {split} prediction diagnostic ===")
    print(json.dumps(report, indent=2))
    (WORK_DIR / f"spectra_siam_{RUN_TAG}_{split}_prediction_diagnostic.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


def metric_summary(labels: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    labels = np.asarray(labels, dtype=np.int64); predicted = (np.asarray(probabilities) >= threshold).astype(np.int64)
    tp = int(((predicted == 1) & (labels == 1)).sum()); fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum()); fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp); recall = tp / max(1, tp + fn); f1 = 2 * precision * recall / max(1e-12, precision + recall)
    specificity = tn / max(1, tn + fp); negative_precision = tn / max(1, tn + fn); negative_f1 = 2 * negative_precision * specificity / max(1e-12, negative_precision + specificity)
    return {"Precision": precision, "Recall": recall, "F1": f1, "Accuracy": (tp + tn) / max(1, len(labels)), "MacroF1": 0.5 * (f1 + negative_f1), "BalancedAccuracy": 0.5 * (recall + specificity), "Threshold": float(threshold), "TP": tp, "FP": fp, "TN": tn, "FN": fn, "Pairs": int(len(labels))}



def save_spectra_language_breakdown(
    frame: pd.DataFrame,
    probabilities: np.ndarray,
    threshold: float,
) -> tuple[Path, pd.DataFrame]:
    """Save test metrics by unordered endpoint-language pair plus ALL."""
    required = {"left_language", "right_language", "label"}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise RuntimeError(f"Language breakdown requires pair columns: {missing}")
    probabilities = np.asarray(probabilities, dtype=np.float64).reshape(-1)
    labels = frame["label"].to_numpy(dtype=np.int64)
    if len(probabilities) != len(frame):
        raise RuntimeError(
            f"Language breakdown received {len(probabilities)} probabilities for {len(frame)} test pairs"
        )

    left_languages = frame["left_language"].astype(str).str.strip().str.lower().tolist()
    right_languages = frame["right_language"].astype(str).str.strip().str.lower().tolist()
    language_keys = np.asarray([
        left if left == right else f"{min(left, right)}->{max(left, right)}"
        for left, right in zip(left_languages, right_languages)
    ], dtype=object)

    rows = []
    groups = [(key, language_keys == key) for key in sorted(set(language_keys.tolist()))]
    groups.append(("ALL", np.ones(len(frame), dtype=bool)))
    for language, mask in groups:
        group_labels = labels[mask]
        group_probabilities = probabilities[mask]
        metrics = metric_summary(group_labels, group_probabilities, threshold)
        rows.append({
            "Dataset": DATASET_KEY,
            "Method": "SPECTRA-Siam",
            "MethodVariant": METHOD_VARIANT,
            "GraphType": "learned_latent",
            "Language": language,
            "LanguageScope": (
                "all" if language == "ALL" else "cross_language" if "->" in language else "same_language"
            ),
            "P": metrics["Precision"],
            "R": metrics["Recall"],
            "F1": metrics["F1"],
            "Acc": metrics["Accuracy"],
            "MacroF1": metrics["MacroF1"],
            "BalancedAccuracy": metrics["BalancedAccuracy"],
            "Threshold": metrics["Threshold"],
            "TP": metrics["TP"],
            "FP": metrics["FP"],
            "TN": metrics["TN"],
            "FN": metrics["FN"],
            "Pairs": metrics["Pairs"],
            "Positives": int((group_labels == 1).sum()),
            "Negatives": int((group_labels == 0).sum()),
        })

    breakdown = pd.DataFrame(rows)
    output_path = WORK_DIR / f"{DATASET_KEY}_{METHOD_VARIANT}_spectra_siam_language_breakdown.csv"
    breakdown.to_csv(output_path, index=False)
    print("Language breakdown:")
    display(breakdown[["Language", "LanguageScope", "P", "R", "F1", "Acc", "Pairs", "Positives"]])
    print("Saved language breakdown:", output_path)
    return output_path, breakdown

def choose_threshold(
    labels: np.ndarray,
    probabilities: np.ndarray,
    selection_metric: str,
) -> tuple[float, dict]:
    if selection_metric not in {"Accuracy", "F1"}:
        raise ValueError(f"Unsupported validation selection metric: {selection_metric}")
    candidates = np.unique(
        np.concatenate(
            (
                np.linspace(0.01, 0.99, 199),
                np.quantile(probabilities, np.linspace(0.0, 1.0, 201)),
            )
        )
    )
    tie_breakers = (
        ("F1", "BalancedAccuracy", "MacroF1")
        if selection_metric == "Accuracy"
        else ("BalancedAccuracy", "MacroF1", "Accuracy")
    )
    ranking = (selection_metric, *tie_breakers)
    best = None
    for threshold in candidates:
        metrics = metric_summary(labels, probabilities, float(threshold))
        score = tuple(metrics[name] for name in ranking)
        if best is None or score > best[2]:
            best = (float(threshold), metrics, score)
    return best[0], best[1]

def predict(model, loader, temperature: float) -> tuple[np.ndarray, np.ndarray]:
    model.eval(); labels_all, probabilities_all = [], []
    with torch.no_grad():
        for left, right, labels in tqdm(loader, desc="Evaluating", leave=False):
            logits, _ = model(left.to(DEVICE), right.to(DEVICE), temperature)
            labels_all.append(labels.numpy()); probabilities_all.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(labels_all), np.concatenate(probabilities_all)


def get_cap(value): return None if value is None or str(value).lower() in {"none", "all", "0"} else int(value)


def prepare_experiment_data() -> tuple[dict[str, pd.DataFrame], dict]:
    corpus = CleanDataCorpus(CLEAN_DATA_DIR)
    limits = {"train": get_cap(MAX_TRAIN_PAIRS), "valid": get_cap(MAX_VALID_PAIRS), "test": get_cap(MAX_TEST_PAIRS)}
    frames = {split: corpus.pairs(split, maximum=limits[split], seed=SEED + offset) for offset, split in enumerate(("train", "valid", "test"))}
    graphs = corpus.load_graphs(
        pair_code_ids(*frames.values()), max_nodes=MAX_AST_NODES,
        strip_node_types=STRIP_NODE_TYPES, strip_topology=STRIP_TOPOLOGY,
    )
    if RUN_DIAGNOSTICS:
        DIAGNOSTIC_REPORTS["input"] = report_input_diagnostics(frames, graphs)
    for split, frame in frames.items():
        frames[split] = frame[frame.left_id.astype(str).isin(graphs) & frame.right_id.astype(str).isin(graphs)].reset_index(drop=True)
        if set(frames[split].label.astype(int).unique()) != {0, 1}: raise RuntimeError(f"{split} has no usable two-class pairs after graph joining")
    print({split: {"pairs": len(frame), "labels": frame.label.value_counts().sort_index().to_dict(), "languages": sorted(set(frame.left_language) | set(frame.right_language))} for split, frame in frames.items()})
    return frames, graphs


def run_experiment() -> dict:
    set_seed(SEED); frames, graphs = prepare_experiment_data()
    loaders = {split: make_loader(frames[split], graphs, max_nodes=MAX_AST_NODES, batch_size=BATCH_SIZE, shuffle=(split == "train"), num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda") for split in ("train", "valid", "test")}
    config = CanonicalSpectraConfig(hidden_dim=HIDDEN_DIM, latent_nodes=LATENT_NODES, slot_iterations=SLOT_ITERATIONS, attention_heads=ATTENTION_HEADS, lexical_dropout=LEXICAL_DROPOUT, use_node_lexical=USE_NODE_LEXICAL, use_source_lexical=USE_SOURCE_LEXICAL, readout_mode=READOUT_MODE, target_density=TARGET_DENSITY, chebyshev_degree=CHEBYSHEV_DEGREE, relation_indices=INPUT_RELATION_INDICES)
    model = CanonicalSpectraSiam(config).to(DEVICE)
    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(f"trainable parameters: {trainable_parameters:,}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED) if hasattr(torch, "amp") else torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    accumulation = max(1, math.ceil(EFFECTIVE_BATCH_SIZE / BATCH_SIZE)); positives = int(frames["train"].label.sum()); negatives = len(frames["train"]) - positives
    positive_weight = min(10.0, negatives / max(1, positives)) if POSITIVE_CLASS_WEIGHT == "auto" else float(POSITIVE_CLASS_WEIGHT)
    validation_positives = int(frames["valid"].label.sum())
    validation_negatives = int(len(frames["valid"]) - validation_positives)
    if validation_positives == 0 or validation_negatives == 0:
        raise RuntimeError("Validation split must contain both clone and non-clone pairs")
    validation_is_balanced = validation_positives == validation_negatives
    validation_selection_metric = "Accuracy" if validation_is_balanced else "F1"
    validation_label_counts = {"clone": validation_positives, "nonclone": validation_negatives}
    print(
        "validation checkpoint policy:",
        {"balanced": validation_is_balanced, "selection_metric": validation_selection_metric, "labels": validation_label_counts},
    )
    STOPPED_EARLY = False
    history, best = [], None; best_path = WORK_DIR / f"spectra_siam_{RUN_TAG}_best.pt"; started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train(); optimizer.zero_grad(set_to_none=True); temperature = None  # encoder infers temperature per graph
        total_loss = 0.0; seen = 0
        for index, (left, right, labels) in enumerate(tqdm(loaders["train"], desc=f"Epoch {epoch}/{EPOCHS} train", leave=False), start=1):
            labels = labels.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                logits, auxiliary = model(left.to(DEVICE), right.to(DEVICE), temperature)
                loss, _ = canonical_spectra_loss(
                    logits, auxiliary, labels,
                    positive_class_weight=positive_weight,
                    spectral_weight=SPECTRAL_WEIGHT, auc_ranking_weight=AUC_RANKING_WEIGHT,
                    reconstruction_weight=RECONSTRUCTION_WEIGHT,
                    graph_weight=GRAPH_WEIGHT,
                    embedding_contrastive_weight=EMBEDDING_CONTRASTIVE_WEIGHT,
                    embedding_negative_margin=EMBEDDING_NEGATIVE_MARGIN,
                    hard_negative_weight=HARD_NEGATIVE_WEIGHT, hard_negative_margin=HARD_NEGATIVE_MARGIN, hard_negative_fraction=HARD_NEGATIVE_FRACTION,
                )
            scaler.scale(loss / accumulation).backward()
            if index % accumulation == 0 or index == len(loaders["train"]):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            total_loss += float(loss.detach().cpu()) * len(labels); seen += len(labels)
        valid_labels, valid_probabilities = predict(model, loaders["valid"], temperature)
        if RUN_DIAGNOSTICS:
            DIAGNOSTIC_REPORTS[f"valid_epoch_{epoch}"] = report_prediction_diagnostics(model, loaders["valid"], temperature, f"valid_epoch_{epoch}")
        threshold, valid = choose_threshold(valid_labels, valid_probabilities, validation_selection_metric)
        row = {"Epoch": epoch, "TrainLoss": total_loss / max(1, seen), **{f"Valid_{k}": v for k, v in valid.items()}}; history.append(row); print(f"epoch={epoch}/{EPOCHS} loss={row['TrainLoss']:.5f} valid_F1={valid['F1']:.4f} threshold={threshold:.4f}")
        if SESSION_BUDGET_HOURS is not None:
            _elapsed_h = (time.perf_counter() - started) / 3600.0
            _per_epoch_h = _elapsed_h / max(epoch, 1)
            if epoch < EPOCHS and _elapsed_h + _per_epoch_h > SESSION_BUDGET_HOURS:
                print(f"[time-guard] {_elapsed_h:.2f}h used, ~{_per_epoch_h:.2f}h per epoch; "
                      f"stopping after epoch {epoch}/{EPOCHS} to stay inside "
                      f"{SESSION_BUDGET_HOURS}h and keep the best checkpoint.")
                STOPPED_EARLY = True
                break
        selection_ranking = (
            (validation_selection_metric, "F1", "BalancedAccuracy", "MacroF1")
            if validation_selection_metric == "Accuracy"
            else (validation_selection_metric, "BalancedAccuracy", "MacroF1", "Accuracy")
        )
        selection_score = tuple(valid[name] for name in selection_ranking)
        best_score = None if best is None else tuple(
            best["valid"][name] for name in selection_ranking
        )
        if best_score is None or selection_score > best_score:
            best = {
                "epoch": epoch,
                "temperature": None,
                "threshold": threshold,
                "valid": valid,
                "selection_metric": validation_selection_metric,
                "validation_is_balanced": validation_is_balanced,
                "validation_label_counts": validation_label_counts,
            }
            torch.save({"model_state": model.state_dict(), "config": asdict(config), "best": best}, best_path)
    checkpoint = torch.load(best_path, map_location=DEVICE); model.load_state_dict(checkpoint["model_state"])
    best_temperature = checkpoint["best"].get("temperature", None)
    test_labels, test_probabilities = predict(model, loaders["test"], best_temperature)
    if RUN_DIAGNOSTICS:
        DIAGNOSTIC_REPORTS["test"] = report_prediction_diagnostics(model, loaders["test"], best_temperature, "test")
    test = metric_summary(test_labels, test_probabilities, checkpoint["best"]["threshold"])
    final_path = WORK_DIR / f"spectra_siam_{RUN_TAG}_final.pt"; torch.save({**checkpoint, "test": test}, final_path)
    history_frame = pd.DataFrame(history); history_frame.to_csv(WORK_DIR / f"spectra_siam_{RUN_TAG}_history.csv", index=False)
    pd.DataFrame([{ "Split": "valid", **checkpoint["best"]["valid"]}, {"Split": "test", **test}]).to_csv(WORK_DIR / f"spectra_siam_{RUN_TAG}_metrics.csv", index=False)
    predictions = frames["test"][["left_id", "right_id", "label"]].copy(); predictions["clone_probability"] = test_probabilities; predictions["prediction"] = (test_probabilities >= checkpoint["best"]["threshold"]).astype(np.int8); predictions.to_csv(WORK_DIR / f"spectra_siam_{RUN_TAG}_predictions.csv.gz", index=False, compression="gzip")
    language_breakdown_path, language_breakdown_frame = save_spectra_language_breakdown(
        frames["test"], test_probabilities, checkpoint["best"]["threshold"]
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 4)); axes[0].plot(history_frame.Epoch, history_frame.TrainLoss, marker="o"); axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Loss"); axes[1].plot(history_frame.Epoch, history_frame.Valid_F1, marker="o", label="F1"); axes[1].plot(history_frame.Epoch, history_frame.Valid_MacroF1, marker="o", label="Macro F1"); axes[1].set(title="Validation metrics", xlabel="Epoch", ylabel="Score"); axes[1].legend(); fig.tight_layout(); fig.savefig(WORK_DIR / f"spectra_siam_{RUN_TAG}_curves.png", dpi=160); plt.show()
    elapsed_seconds = float(time.perf_counter() - started)
    result = {"dataset": DATASET_KEY, "validation_selection_metric": validation_selection_metric, "validation_is_balanced": validation_is_balanced, "best_epoch": checkpoint["best"]["epoch"], "best_valid": checkpoint["best"]["valid"], "test": test, "seconds": elapsed_seconds, "RuntimeSeconds": elapsed_seconds, "RuntimeMinutes": elapsed_seconds / 60.0, "TrainableParameters": int(trainable_parameters), "checkpoint": str(final_path)}
    if RUN_DIAGNOSTICS:
        result["diagnostics"] = DIAGNOSTIC_REPORTS
    result["run_metadata"] = {
        "RunProfile": RUN_PROFILE,
        "ValidationSelectionMetric": validation_selection_metric,
        "ValidationBalanced": validation_is_balanced,
        "ValidationLabelCounts": validation_label_counts,
        "Seed": int(SEED),
        "ConfiguredHyperparameters": {
            "MaxTrainPairs": MAX_TRAIN_PAIRS, "MaxValidPairs": MAX_VALID_PAIRS, "MaxTestPairs": MAX_TEST_PAIRS,
            "Epochs": EPOCHS, "BatchSize": BATCH_SIZE, "EffectiveBatchSize": EFFECTIVE_BATCH_SIZE,
            "LearningRate": LEARNING_RATE, "WeightDecay": WEIGHT_DECAY, "MaxAstNodes": MAX_AST_NODES,
            "HiddenDim": HIDDEN_DIM, "LatentNodes": LATENT_NODES, "SlotIterations": SLOT_ITERATIONS,
            "AttentionHeads": ATTENTION_HEADS, "LexicalDropout": LEXICAL_DROPOUT,
            "TargetDensity": TARGET_DENSITY, "ChebyshevDegree": CHEBYSHEV_DEGREE,
            "SpectralWeight": SPECTRAL_WEIGHT, "AucRankingWeight": AUC_RANKING_WEIGHT, "ReconstructionWeight": RECONSTRUCTION_WEIGHT,
            "GraphWeight": GRAPH_WEIGHT, "PositiveClassWeight": POSITIVE_CLASS_WEIGHT,
            "TemperatureStart": TEMPERATURE_START, "TemperatureEnd": TEMPERATURE_END,
            "InputRelations": list(INPUT_RELATION_INDICES), "AdaptiveTemperature": USE_ADAPTIVE_TEMPERATURE, "AdaptiveTemperatureMin": ADAPTIVE_TEMPERATURE_MIN, "AdaptiveTemperatureMax": ADAPTIVE_TEMPERATURE_MAX,
            "SpectralAttention": USE_SPECTRAL_ATTENTION, "HardNegativeWeight": HARD_NEGATIVE_WEIGHT, "HardNegativeMargin": HARD_NEGATIVE_MARGIN, "HardNegativeFraction": HARD_NEGATIVE_FRACTION,
            "UseNodeLexical": USE_NODE_LEXICAL, "UseSourceLexical": USE_SOURCE_LEXICAL, "ReadoutMode": READOUT_MODE,
        },
        "Environment": {
            "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "GPUCapability": ".".join(map(str, torch.cuda.get_device_capability(0))) if torch.cuda.is_available() else None,
            "TorchVersion": torch.__version__,
        },
        "CompletedUTC": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
    }
    # A flat result CSV intentionally matches the baseline-result schema.
    # It makes final-paper tables a direct concat rather than a manual conversion.
    comparison_row = {
        "Dataset": DATASET_KEY.upper(),
        "Method": "SPECTRA-Siam",
        "MethodVariant": f"{METHOD_VARIANT} / AST+DDG / adaptive-temperature / spectral-attention / batch-hard-negatives",
        "RunProfile": RUN_PROFILE,
        "ValidationSelectionMetric": validation_selection_metric,
        "ValidationBalanced": validation_is_balanced,
        "ValidationLabelCounts": validation_label_counts,
        "Seed": int(SEED),
        "BestEpoch": checkpoint["best"]["epoch"],
        "BestValidF1": checkpoint["best"]["valid"]["F1"],
        "P": test["Precision"], "R": test["Recall"], "F1": test["F1"], "Acc": test["Accuracy"],
        "MacroF1": test["MacroF1"], "BalancedAccuracy": test["BalancedAccuracy"],
        "Threshold": test["Threshold"],
        "TP": test["TP"], "FP": test["FP"], "TN": test["TN"], "FN": test["FN"],
        "TrainPairs": len(frames["train"]), "ValidPairs": len(frames["valid"]), "TestPairs": len(frames["test"]),
        "ConfiguredEpochs": EPOCHS, "BatchSize": BATCH_SIZE, "EffectiveBatchSize": EFFECTIVE_BATCH_SIZE,
        "LearningRate": LEARNING_RATE, "WeightDecay": WEIGHT_DECAY,
        "InputRelations": "AST+DDG", "AdaptiveTemperature": USE_ADAPTIVE_TEMPERATURE, "SpectralAttention": USE_SPECTRAL_ATTENTION, "HardNegativeWeight": HARD_NEGATIVE_WEIGHT,
        "TrainableParameters": int(trainable_parameters),
        "RuntimeSeconds": elapsed_seconds, "RuntimeMinutes": elapsed_seconds / 60.0,
        "GPU": result["run_metadata"]["Environment"]["GPU"],
        "GPUCapability": result["run_metadata"]["Environment"]["GPUCapability"],
        "TorchVersion": result["run_metadata"]["Environment"]["TorchVersion"],
        "CompletedUTC": result["run_metadata"]["CompletedUTC"],
    }
    comparison_path = WORK_DIR / f"{RUN_TAG}_spectra_siam_results.csv"
    pd.DataFrame([comparison_row]).to_csv(comparison_path, index=False)
    result["comparison_csv"] = str(comparison_path)
    result["language_breakdown_csv"] = str(language_breakdown_path)
    result["language_breakdown_rows"] = int(len(language_breakdown_frame))
    result["run_metadata"]["ActualGraphEvaluablePairs"] = {
        "train": len(frames["train"]), "valid": len(frames["valid"]), "test": len(frames["test"]),
    }
    result["run_metadata"]["OutputFiles"] = {
        "comparison_results": str(comparison_path),
        "metrics": str(WORK_DIR / f"spectra_siam_{RUN_TAG}_metrics.csv"),
        "history": str(WORK_DIR / f"spectra_siam_{RUN_TAG}_history.csv"),
        "predictions": str(WORK_DIR / f"spectra_siam_{RUN_TAG}_predictions.csv.gz"),
        "language_breakdown": str(language_breakdown_path),
        "checkpoint": str(final_path),
    }
    metadata_path = WORK_DIR / f"{RUN_TAG}_spectra_siam_run_metadata.json"
    metadata_path.write_text(json.dumps(result["run_metadata"], indent=2, default=str), encoding="utf-8")
    result["run_metadata_path"] = str(metadata_path)
    (WORK_DIR / f"spectra_siam_{RUN_TAG}_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("Comparable SPECTRA result:", comparison_path)
    print("Research metadata:", metadata_path)
    return result


In [ ]:
if RUN_EXPERIMENT:
    RESULTS = run_experiment()
    display(pd.DataFrame([RESULTS["best_valid"], RESULTS["test"]], index=["best_valid", "test"]))
    print(json.dumps(RESULTS, indent=2))
else:
    print("RUN_EXPERIMENT=False; configuration and implementation cells loaded without training.")
